# 8x8um bin preprocessing
Bin-level analogue of `CustomBinning.ipynb` -- same QC -> clustering -> patch-assignment -> merge structure, but for native (or aggregated) Visium HD 8um grid bins instead of StarDist nucleus segmentation. Much shorter than the cell notebook: bins need no polygon loading, no spatial join into a segmented shape, and no side-car geometry file -- `functions/label_transfer.py`'s `load_grid_bin_counts`/`aggregate_expression_to_grid` already hand back an AnnData with `obsm['spatial']` populated directly.

Output sits as a clean sibling of the cell-level tree: `data/{sample}/01_preprocessing/grid_bin/008um/custom_binning/count_area_filtered_adata.h5ad` (same filename as the cell path, different `dir_cellseg`-equivalent directory: `grid_bin` instead of `cell_seg_stardist`) -- deliberately matching the cell path's naming/column conventions (`Slide_ID`, `id`, `Patch_ID`) so a later autoencoder-side integration (out of scope here) is a path change, not a code change; see `functions/dual_autoencoder.py`'s `build_pairing`.

In [ ]:
from pathlib import Path
import gc
import sys

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

sys.path.insert(0, str(Path("../..").resolve()))
from functions.label_transfer import load_grid_bin_counts, aggregate_expression_to_grid, assign_bins_to_patches
from functions.merge import read_h5ad_with_null_uns_healing, sanitize_uns

## Config
Adapt `sample` per run, same convention as `CustomBinning.ipynb`.

In [ ]:
sample = "CRC_P1"   # adapt to requirements
dir_binning = "grid_bin"   # sibling to "cell_seg_stardist" -- no segmentation, just native/aggregated square bins
bin_size = "008um"         # matches functions/label_transfer.py's GRID_SPATIAL_DIR_TEMPLATE convention
group_size = 4             # 2um -> 8um aggregation factor, forwarded to aggregate_expression_to_grid
umi_cutoff = 80             # same starting point as the cell path -- tune via the diagnostic plot below

basedir = Path("../../data")
dir_result_output = basedir / sample / "01_preprocessing" / dir_binning / bin_size
# "custom_binning" leaf name kept for template parity with the cell path -- CustomBinning.ipynb's
# merge cells (mirrored near the end of this notebook) become a straight path-value swap instead of
# a template-string restructure.
dir_custom_binning = dir_result_output / "custom_binning"
dir_result_output.mkdir(parents=True, exist_ok=True)
dir_custom_binning.mkdir(parents=True, exist_ok=True)

## Load bin counts
Same runtime check `main_grid()` already uses in `functions/label_transfer.py`: native SpaceRanger `square_008um` output if present, else exact 4x4 aggregation from `square_002um` (not an approximation -- Visium HD's bin grid is hierarchically nested, see `aggregate_expression_to_grid`'s docstring).

In [ ]:
grid_h5_path = basedir / sample / "output/binned_outputs/square_008um/filtered_feature_bc_matrix.h5"
if grid_h5_path.exists():
    print(f"{sample}: native square_008um output found")
    adata = load_grid_bin_counts(sample, data_dir=basedir)
else:
    print(f"{sample}: no native square_008um output -- aggregating from square_002um instead")
    adata = aggregate_expression_to_grid(sample, group_size=group_size, data_dir=basedir)

# Neither loader sets Slide_ID or a canonical, non-sample-prefixed obs id -- add both here so this
# file's obs schema matches the cell path's (Slide_ID, id) exactly. Also normalizes obs_names to the
# SAME (un-prefixed) convention regardless of which loader ran: load_grid_bin_counts leaves raw,
# sample-collidable 10x barcodes as obs_names, while aggregate_expression_to_grid already returns
# sample-prefixed ids -- without this, merge_anndatas(..., obs_name_mode="prefix") later would
# double-prefix whichever loader's output took this path.
adata.obs["Slide_ID"] = sample
adata.obs["id"] = [f"grid_{r}_{c}" for r, c in zip(adata.obs["array_row"], adata.obs["array_col"])]
adata.obs_names = adata.obs["id"].values
adata.var_names_make_unique()
print(f"{sample}: {adata.n_obs} bins, {adata.n_vars} genes")

## QC
UMI-count only -- no area filter (every bin is the same fixed 8x8um size, so nucleus-style area QC is meaningless here). Reuses the cell notebook's own generic UMI-cutoff helpers verbatim (both are notebook-local functions with no nucleus/area assumptions in either).

In [ ]:
from scipy.stats import median_abs_deviation

def total_umi(adata_, cut_off):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].boxplot(adata_.obs["total_counts"], vert=False, widths=0.7, patch_artist=True, boxprops=dict(facecolor='skyblue'))
    axs[0].set_title('Total Counts')
    axs[1].boxplot(adata_.obs["total_counts"][adata_.obs["total_counts"] > cut_off], vert=False, widths=0.7, patch_artist=True, boxprops=dict(facecolor='skyblue'))
    axs[1].set_title('Total Counts > ' + str(cut_off))
    for ax in axs:
        ax.get_yaxis().set_visible(False)
    plt.tight_layout()
    plt.show()

def plot_umi_cutoff_diagnostics(adata_, current_cutoff):
    counts = adata_.obs["total_counts"].values
    log_counts = np.log1p(counts)

    sorted_counts = np.sort(counts)[::-1]
    x = np.arange(len(sorted_counts))
    y = np.log1p(sorted_counts)
    x_norm = (x - x.min()) / (x.max() - x.min())
    y_norm = (y - y.min()) / (y.max() - y.min())
    line_vec = np.array([x_norm[-1] - x_norm[0], y_norm[-1] - y_norm[0]])
    line_vec_norm = line_vec / np.linalg.norm(line_vec)
    points = np.column_stack([x_norm - x_norm[0], y_norm - y_norm[0]])
    proj_lengths = points @ line_vec_norm
    proj_points = np.outer(proj_lengths, line_vec_norm)
    distances = np.linalg.norm(points - proj_points, axis=1)
    knee_idx = np.argmax(distances)
    knee_cutoff = sorted_counts[knee_idx]

    med = np.median(log_counts)
    mad = median_abs_deviation(log_counts, scale="normal")
    mad_cutoff_log = med - 3 * mad
    mad_cutoff = max(np.expm1(mad_cutoff_log), 0)

    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    axs[0].hist(log_counts, bins=80, edgecolor='black', alpha=0.7)
    axs[0].axvline(np.log1p(current_cutoff), color='red', linestyle='--', label=f'Current cutoff: {current_cutoff}')
    axs[0].axvline(np.log1p(knee_cutoff), color='green', linestyle='--', label=f'Knee cutoff: {knee_cutoff:.0f}')
    axs[0].axvline(mad_cutoff_log, color='purple', linestyle='--', label=f'MAD cutoff (3 MADs): {mad_cutoff:.0f}')
    axs[0].set_xlabel('log1p(total_counts)'); axs[0].set_ylabel('Number of bins'); axs[0].legend()
    axs[1].plot(x, y, color='black', linewidth=1)
    axs[1].axhline(np.log1p(current_cutoff), color='red', linestyle='--', label=f'Current cutoff: {current_cutoff}')
    axs[1].axvline(knee_idx, color='green', linestyle='--', label=f'Knee point (rank {knee_idx})')
    axs[1].axhline(mad_cutoff_log, color='purple', linestyle='--', label=f'MAD cutoff: {mad_cutoff:.0f}')
    axs[1].set_xlabel('Bin rank (sorted by total_counts, descending)'); axs[1].set_ylabel('log1p(total_counts)'); axs[1].legend()
    plt.tight_layout()
    plt.show()

    n_total = len(counts)
    print(f"Total bins: {n_total}")
    print(f"  Pass current cutoff (>{current_cutoff}): {(counts > current_cutoff).sum()} ({(counts > current_cutoff).mean():.1%})")
    print(f"  Pass knee cutoff (>{knee_cutoff:.0f}):    {(counts > knee_cutoff).sum()} ({(counts > knee_cutoff).mean():.1%})")
    print(f"  Pass MAD cutoff (>{mad_cutoff:.0f}):      {(counts > mad_cutoff).sum()} ({(counts > mad_cutoff).mean():.1%})")
    return knee_cutoff, mad_cutoff

sc.pp.calculate_qc_metrics(adata, inplace=True)
total_umi(adata, umi_cutoff)
knee_cutoff, mad_cutoff = plot_umi_cutoff_diagnostics(adata, umi_cutoff)

In [ ]:
mask_count = adata.obs["total_counts"] > umi_cutoff
count_area_filtered_adata = adata[mask_count].copy()   # name kept identical to the cell path's variable/file name deliberately
count_area_filtered_adata.var_names_make_unique()
sc.pp.calculate_qc_metrics(count_area_filtered_adata, inplace=True)
print(f"{mask_count.sum()}/{adata.n_obs} bins kept (total_counts > {umi_cutoff})")

## Clustering
Normalize -> log1p -> HVG -> PCA -> neighbors -> Leiden, copied verbatim from `CustomBinning.ipynb`'s own clustering cell -- 100% generic scanpy calls, no cell/nucleus assumptions. Unlike the cell path, there's no `.obs['geometry']` column to strip before writing (bins have no polygon at all).

In [ ]:
sc.pp.normalize_total(count_area_filtered_adata, inplace=True)
sc.pp.log1p(count_area_filtered_adata)
sc.pp.highly_variable_genes(count_area_filtered_adata, flavor="seurat", n_top_genes=2000)
sc.pp.pca(count_area_filtered_adata)
sc.pp.neighbors(count_area_filtered_adata)
sc.tl.leiden(count_area_filtered_adata, resolution=0.2, key_added="clusters",
             flavor="igraph", n_iterations=2, directed=False)  # default leidenalg backend is
             # impractically slow at bin scale (systematically more nodes than the cell path,
             # since every tissue region gets a bin regardless of segmentation success) --
             # igraph is the convention already used elsewhere in this codebase

count_area_filtered_adata.write(dir_custom_binning / "count_area_filtered_adata.h5ad")
print(f"wrote {dir_custom_binning / 'count_area_filtered_adata.h5ad'}")

## Assigning bins to histology patches
Assigns each bin a `Patch_ID` so the output satisfies `build_pairing`'s contract (`functions/dual_autoencoder.py`) for a later integration. Reuses `functions.label_transfer.assign_bins_to_patches` (a cheap floor-division lookup, already used there for the analogous 2um-bin case) rather than the cell path's `gpd.overlay` polygon-intersection machinery -- an 8um bin is ~2 orders of magnitude smaller than a 224px patch and essentially never straddles a patch boundary meaningfully, so exact overlap-percentage bookkeeping buys nothing here.

In [ ]:
exclude_dirs = ["new", "archived"]
samples = [p.name for p in basedir.iterdir() if (p.is_dir() and p.name != "combined" and p.name not in exclude_dirs)]
print("Samples to assign patches for:", samples)

for s in samples:
    dir_trans = basedir / s / "01_preprocessing" / dir_binning / bin_size / "custom_binning"
    dir_histology = basedir / s / "01_preprocessing/unimodel/metadata"

    if not (dir_histology / "filtered_WSI_patch_embeddings.h5ad").exists():
        print(f"Skipping {s}: no filtered_WSI_patch_embeddings.h5ad yet.")
        continue
    if not (dir_trans / "count_area_filtered_adata.h5ad").exists():
        print(f"Skipping {s}: no bin-level count_area_filtered_adata.h5ad yet (run the cells above for it first).")
        continue

    print(f"\n=== {s} ===")
    adata_patches = sc.read_h5ad(dir_histology / "filtered_WSI_patch_embeddings.h5ad")
    bin_adata = read_h5ad_with_null_uns_healing(dir_trans / "count_area_filtered_adata.h5ad")

    tissue_positions = pd.DataFrame(
        {"pxl_col_in_fullres": bin_adata.obsm["spatial"][:, 0],
         "pxl_row_in_fullres": bin_adata.obsm["spatial"][:, 1]},
        index=bin_adata.obs_names,
    )
    patch_ids = assign_bins_to_patches(tissue_positions, adata_patches.obs)
    bin_adata.obs["Patch_ID"] = patch_ids.reindex(bin_adata.obs_names)

    n_missing = bin_adata.obs["Patch_ID"].isna().sum()
    print(f"{s}: {n_missing}/{bin_adata.n_obs} bins got no Patch_ID (outside any extracted patch)")

    stray_nones = sanitize_uns(bin_adata)
    if stray_nones:
        print(f"{s}: sanitized stray null uns entries before saving: {stray_nones}")
    bin_adata.write(dir_trans / "count_area_filtered_adata.h5ad")

## Merging the output
Run after bin preprocessing (load -> QC -> clustering -> patch assignment) has been done for every sample. `functions/merge.py`'s `merge_anndatas` is fully generic (only needs a `Slide_ID` obs column) -- reused unchanged, just pointed at the new `grid_bin/008um` path template. Only one merge call is needed here (unlike the cell path's four): bins have no two-stage aggregation (`nuclei_grouped_UMI_count.h5ad`/`filtered_custom_bins.h5ad` have no bin analogue) and no separate geometry parquet to merge.

In [ ]:
import sys
sys.path.insert(0, str(Path("../..").resolve()))
from functions.merge import merge_anndatas

dir_grid_binning = "{basedir}/{sample}/01_preprocessing/grid_bin/{bin_size}/custom_binning"
path_template_grid_binning = dir_grid_binning + "/{filename}"

exclude_dirs = ["new", "archived"]
samples = [p.name for p in basedir.iterdir() if (p.is_dir() and p.name != "combined" and p.name not in exclude_dirs)]
print("Samples to merge:", samples)

REFERENCE_ONLY_VAR_COLS = {"gene_ids", "feature_types", "genome"}

In [ ]:
print("merging bin-level count_area_filtered_adata")
merged_count_area_filtered_adata = merge_anndatas(
    samples,
    path_template_grid_binning,
    "count_area_filtered_adata.h5ad",
    basedir,
    auto_detect_var_cols=True,
    reference_only_var_cols=REFERENCE_ONLY_VAR_COLS,
    bin_size=bin_size,
    per_sample_uns_moran_key="moranI",
    merge_mode="full",
)
del merged_count_area_filtered_adata
gc.collect()

## Recalculating global SVG, HVG, PCA
Copied unchanged from `CustomBinning.ipynb`'s own recompute cell -- generic over the `varm`/`uns` structure `merge_anndatas` produces, regardless of what unit was merged. `varm['moranI_per_sample']` is only populated once `get_svg.py --unit bin` has been run for every sample (it writes back to the canonical `count_area_filtered_adata.h5ad` this merge just read -- rerun this cell after that).

In [ ]:
f = dir_grid_binning.format(basedir=basedir, sample="combined", bin_size=bin_size)
merged_count_area_filtered_adata = sc.read_h5ad(Path(f) / "count_area_filtered_adata.h5ad")

hvg_per_sample = np.nan_to_num(
    merged_count_area_filtered_adata.varm['highly_variable_per_sample'], nan=0.0
).astype(bool)
merged_count_area_filtered_adata.var['highly_variable'] = hvg_per_sample.any(axis=1)
merged_count_area_filtered_adata.var['highly_variable_n_samples'] = hvg_per_sample.sum(axis=1)

I_THRESH = 0.1
FDR_THRESH = 0.05
if 'moranI_per_sample' in merged_count_area_filtered_adata.varm:
    moranI = merged_count_area_filtered_adata.varm['moranI_per_sample']
    moranI_pval = merged_count_area_filtered_adata.varm['moranI_pval_per_sample']
    svg_per_sample = (moranI_pval < FDR_THRESH) & (moranI > I_THRESH)
    svg_per_sample = np.nan_to_num(svg_per_sample, nan=0.0).astype(bool)
    merged_count_area_filtered_adata.var['spatially_variable'] = svg_per_sample.any(axis=1)
    merged_count_area_filtered_adata.var['spatially_variable_n_samples'] = svg_per_sample.sum(axis=1)
else:
    print("No moranI_per_sample yet -- run get_svg.py --unit bin for every sample, then re-run the merge cell above, then this one.")
    merged_count_area_filtered_adata.var['spatially_variable'] = False
    merged_count_area_filtered_adata.var['spatially_variable_n_samples'] = 0

sc.pp.highly_variable_genes(
    merged_count_area_filtered_adata,
    flavor="seurat",
    n_top_genes=2000,
    batch_key="Slide_ID",
)
sc.pp.pca(merged_count_area_filtered_adata, use_highly_variable=True)

merged_count_area_filtered_adata.var['gene_selection'] = 'none'
merged_count_area_filtered_adata.var.loc[
    merged_count_area_filtered_adata.var['highly_variable'], 'gene_selection'
] = 'HVG'
merged_count_area_filtered_adata.var.loc[
    merged_count_area_filtered_adata.var['spatially_variable'], 'gene_selection'
] = 'SVG'
merged_count_area_filtered_adata.var.loc[
    merged_count_area_filtered_adata.var['highly_variable']
    & merged_count_area_filtered_adata.var['spatially_variable'],
    'gene_selection'
] = 'HVG_and_SVG'

stray_nones = sanitize_uns(merged_count_area_filtered_adata)
if stray_nones:
    print("Sanitized stray null uns entries before saving:", stray_nones)
merged_count_area_filtered_adata.write(Path(f) / "count_area_filtered_adata.h5ad")

print(merged_count_area_filtered_adata.var['gene_selection'].value_counts())
print("TOTAL SVG union count:", merged_count_area_filtered_adata.var['spatially_variable'].sum())